# Eora parser walkthrough

This notebook is the practical guide for parsing the Eora datasets supported by MARIO.


## What this notebook covers

- which Eora workflows MARIO supports and which it does not;
- the difference between `Eora26` and single-region local files;
- which files are required for the `Eora26` workflow;
- how `multi_region=`, `indeces=`, `country=`, and `price=` are used;
- how `name_convention=` and `aggregate_trade=` affect the single-region workflow;
- which parser caveats matter in practice.


## Important scope note

MARIO does **not** parse the full Eora MRIO release.

The supported workflows are only:

- `Eora26` as the multi-region workflow;
- local single-region Eora tables.


## Relevant source page

- Official Eora website: [Eora MRIO portal](https://www.worldmrio.com/)

Automatic download is not part of the current MARIO workflow. The expected path is to work from local files that you already downloaded.


## Main entry point

For normal user workflows, the public entry point is:

- `mario.parse_eora(...)`

The same function supports:

- `Eora26` with `multi_region=True`;
- local single-region Eora files with `multi_region=False`.


## Key arguments

The key public arguments are:

- `path`: one Eora file or one local dataset directory;
- `multi_region`: use `True` for `Eora26` and `False` for local single-region files;
- `table`: mainly relevant for single-region parsing;
- `indeces`: optional path to the `Eora26` label files;
- `name_convention`: `full_name` or `abbreviation` in the single-region workflow;
- `aggregate_trade`: whether to collapse detailed import/export rows in the single-region workflow;
- `country`: country selector when a directory contains multiple single-region files;
- `price`: optional selector when a single-region directory contains multiple price variants.


## `Eora26` versus single-region Eora

Use `multi_region=True` only for `Eora26`. This workflow expects the standard `Eora26_<year>_<price>_*.txt` files plus the `labels_*.txt` files.

Use `multi_region=False` for the local single-region tables. In that case MARIO resolves one file such as `IO_ITA_2015_BasicPrice.txt` and can usually infer whether it behaves like an `IOT` or a `SUT`.


## Label files for `Eora26`

For `Eora26`, MARIO needs both:

- the numeric files such as `T`, `FD`, `VA`, `Q`, and `QY`;
- the label files such as `labels_T.txt`, `labels_FD.txt`, `labels_VA.txt`, and `labels_Q.txt`.

If the label files already live next to the dataset files, `indeces=` can be omitted. Otherwise, point `indeces=` to the directory that contains them.


In [ ]:
import mario

## Parse one `Eora26` directory

This is the supported multi-region Eora workflow.


In [ ]:
db = mario.parse_eora(
    path="/path/to/eora26_directory",
    multi_region=True,
    table="IOT",
    indeces="/path/to/eora26_labels",
    calc_all=False,
)

db

## Parse `Eora26` when labels are colocated

If the `labels_*.txt` files already live in the same directory as the numeric files, `indeces=` is not required.


In [ ]:
db = mario.parse_eora(
    path="/path/to/eora26_directory",
    multi_region=True,
    table="IOT",
    calc_all=False,
)

db

## Parse one local single-region file

This is the non-multi-region Eora workflow. When `path` points to a directory, `country=` selects the file to parse.


In [ ]:
db = mario.parse_eora(
    path="/path/to/single_region_directory",
    multi_region=False,
    country="ITA",
    calc_all=False,
)

db

## Select one price variant in the single-region workflow

Use `price=` when the directory contains more than one variant for the same country and year.


In [ ]:
db = mario.parse_eora(
    path="/path/to/single_region_directory",
    multi_region=False,
    country="ITA",
    price="BasicPrice",
    calc_all=False,
)

db

## Change the naming convention

In the single-region workflow, `name_convention=` controls whether MARIO uses full country names or abbreviations in the region labels.


In [ ]:
db = mario.parse_eora(
    path="/path/to/single_region_directory",
    multi_region=False,
    country="ITA",
    name_convention="abbreviation",
    calc_all=False,
)

db

## Aggregate trade rows in the single-region workflow

Use `aggregate_trade=True` when you want imports and exports collapsed into totals instead of preserving bilateral trade labels.


In [ ]:
db = mario.parse_eora(
    path="/path/to/single_region_directory",
    multi_region=False,
    country="ITA",
    aggregate_trade=True,
    calc_all=False,
)

db

## Notes and caveats

- there is no parser here for the full Eora MRIO release;
- multi-region means `Eora26` only;
- the `Eora26` parser applies a few consistency repairs during parsing; those notes are stored in metadata;
- single-region parsing can infer `IOT` versus `SUT` automatically from the local file structure.


In [ ]:
db.meta_history
db.sets
db.units